In [1]:
import sys 
import os 

ls = os.path.abspath('../')
sys.path.append(ls)
from utils.sampler import *

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
N_rays = 1400
rays_o = torch.rand(N_rays, 3, device=device)
rays_d = torch.randn(N_rays, 3, device=device)
rays_d = rays_d / torch.norm(rays_d, dim=-1, keepdim=True)
bounds = torch.stack([torch.ones(N_rays, device=device) * 2.0,
                      torch.ones(N_rays, device=device) * 6.0], dim=-1)


In [3]:
coarse = StratifiedSampler(N_samples=64, perturb=1.0).to(device)
pts_coarse, z_vals_coarse = coarse(rays_o, rays_d, bounds)
print(f"Coarse: {pts_coarse.shape}, {z_vals_coarse.shape}")


Coarse: torch.Size([1400, 64, 3]), torch.Size([1400, 64])


## TEST OUR MLP IMPLEMENTATION

In [4]:
from utils.nerfmlp import *
from utils.volumerendering import *
from utils.encoding import *

In [5]:
batch_size = 1024
num_samples = 64
num_rays = batch_size
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"\nDevice: {device}")


Device: cpu


In [6]:
multires = 10
multires_views = 4
input_dim = 3
output_dim = 4

pos_encoder = PositionalEncoding(L=multires, include_input=True)
dir_encoder = PositionalEncoding(L=multires_views, include_input=True)

In [7]:
input_ch = input_dim + 2 * multires * input_dim  # 3 + 2*10*3 = 63
input_ch_views = input_dim + 2 * multires_views * input_dim  # 3 + 2*4*3 = 27
print(f"\nInput dimensions:")
print(f"  Position encoding: {input_ch}")
print(f"  Direction encoding: {input_ch_views}")



Input dimensions:
  Position encoding: 63
  Direction encoding: 27


In [8]:
mlp = MLP(
    D=8,
    W=256,
    input_ch=input_ch,  # Only position encoding
    input_ch_views=input_ch_views,
    output_ch=output_dim,
    skips=[4],
    use_viewdirs=True
).to(device)

In [9]:
mlp.eval()

MLP(
  (pts_linears): ModuleList(
    (0): Linear(in_features=63, out_features=256, bias=True)
    (1-4): 4 x Linear(in_features=256, out_features=256, bias=True)
    (5): Linear(in_features=319, out_features=256, bias=True)
    (6-7): 2 x Linear(in_features=256, out_features=256, bias=True)
  )
  (alpha_linear): Linear(in_features=256, out_features=1, bias=True)
  (feature_linear): Linear(in_features=256, out_features=256, bias=True)
  (views_linears): ModuleList(
    (0): Linear(in_features=283, out_features=128, bias=True)
  )
  (rgb_linear): Linear(in_features=128, out_features=3, bias=True)
)

In [10]:
print(f"\nMLP architecture:")
print(f"  input_ch (positions): {mlp.input_ch}")
print(f"  input_ch_views (directions): {mlp.input_ch_views}")
print(f"  Total parameters: {sum(p.numel() for p in mlp.parameters()):,}")


MLP architecture:
  input_ch (positions): 63
  input_ch_views (directions): 27
  Total parameters: 595,844


In [11]:
pts = torch.randn(num_rays, num_samples, 3, device=device)
dirs = torch.randn(num_rays, num_samples, 3, device=device)
dirs = F.normalize(dirs, dim=-1)

In [12]:
pts_flat = pts.reshape(-1, 3)
dirs_flat = dirs.reshape(-1, 3)

In [13]:
pts_encoded = pos_encoder(pts_flat)
dirs_encoded = dir_encoder(dirs_flat)
print(f"\nEncoded shapes:")
print(f"  Points: {pts_encoded.shape}")
print(f"  Directions: {dirs_encoded.shape}")


Encoded shapes:
  Points: torch.Size([65536, 63])
  Directions: torch.Size([65536, 27])


In [14]:
mlp_input = torch.cat([pts_encoded, dirs_encoded], dim=-1)
print(f"MLP input shape: {mlp_input.shape}")

MLP input shape: torch.Size([65536, 90])


In [15]:
raw_flat = mlp(mlp_input)
print(f"MLP output shape (flat): {raw_flat.shape}")


MLP output shape (flat): torch.Size([65536, 4])


In [16]:
raw = raw_flat.reshape(num_rays, num_samples, output_dim)
print(f"MLP output shape (reshaped): {raw.shape}")

MLP output shape (reshaped): torch.Size([1024, 64, 4])


In [17]:
expected_shape = (num_rays, num_samples, 4)
assert raw.shape == expected_shape, f"❌ Expected {expected_shape}, got {raw.shape}"
print(f"✅ Output shape correct: {raw.shape}")

print(f"\nOutput value ranges:")
print(f"  RGB: [{raw[..., :3].min():.4f}, {raw[..., :3].max():.4f}]")
print(f"  Density: [{raw[..., 3].min():.4f}, {raw[..., 3].max():.4f}]")

print("\n" + "="*80)
print("✅ TEST PASSED - MLP works correctly!")
print("="*80)

✅ Output shape correct: torch.Size([1024, 64, 4])

Output value ranges:
  RGB: [-0.1984, 0.2283]
  Density: [0.0278, 0.0933]

✅ TEST PASSED - MLP works correctly!


In [18]:
print("\n" + "="*80)
print("Test 1: MipMLP (Mip-NeRF)")
print("="*80)

mip_mlp = MipMLP(
    input_dim=3,
    output_dim=4,
    net_depth=8,
    net_width=256,
    skips=[4],
    viewdirs=True,
    multires=10,
    multires_views=4
).to(device)

pos_encoder_mip = IntegratedPositionalEncoding(L=10, num_freqs=10, include_input=True)
dir_encoder_mip = IntegratedPositionalEncoding(L=4, num_freqs=4, include_input=True)

mip_mlp.set_embedder(pos_encoder_mip, dir_encoder_mip)

# Generate test data
pts = torch.randn(num_rays, num_samples, 3, device=device)
pts_cov = torch.randn(num_rays, num_samples, 3, device=device).abs() * 0.01
dirs = torch.randn(num_rays, num_samples, 3, device=device)
dirs = F.normalize(dirs, dim=-1)

# Flatten
pts_flat = pts.reshape(-1, 3)
pts_cov_flat = pts_cov.reshape(-1, 3)
dirs_flat = dirs.reshape(-1, 3)

print(f"\nInput shapes:")
print(f"  Points: {pts_flat.shape}")
print(f"  Covariance: {pts_cov_flat.shape}")
print(f"  Directions: {dirs_flat.shape}")

# Forward pass
raw_flat = mip_mlp(pts_flat, pts_cov_flat, dirs_flat)
print(f"MipMLP output shape (flat): {raw_flat.shape}")

# Reshape
raw = raw_flat.reshape(num_rays, num_samples, 4)
print(f"MipMLP output shape (reshaped): {raw.shape}")

expected_shape = (num_rays, num_samples, 4)
assert raw.shape == expected_shape, f"❌ Expected {expected_shape}, got {raw.shape}"
print(f"✅ Output shape correct: {raw.shape}")

print(f"\nOutput value ranges:")
print(f"  RGB: [{raw[..., :3].min():.4f}, {raw[..., :3].max():.4f}]")
print(f"  Density: [{raw[..., 3].min():.4f}, {raw[..., 3].max():.4f}]")

print(f"\nTotal parameters: {sum(p.numel() for p in mip_mlp.parameters()):,}")


Test 1: MipMLP (Mip-NeRF)

Input shapes:
  Points: torch.Size([65536, 3])
  Covariance: torch.Size([65536, 3])
  Directions: torch.Size([65536, 3])
MipMLP output shape (flat): torch.Size([65536, 4])
MipMLP output shape (reshaped): torch.Size([1024, 64, 4])
✅ Output shape correct: torch.Size([1024, 64, 4])

Output value ranges:
  RGB: [-0.1103, 0.1669]
  Density: [-0.0514, 0.0059]

Total parameters: 595,844


In [19]:
instant_ngp = InstantNGP(
    input_dim=3,
    output_dim=4,
    net_depth=2,
    net_width=64,
    geo_feat_dim=15,
    viewdirs=True,
    num_levels=16,
    features_per_level=2,
    multires_views=4
).to(device)

hash_encoder = HashEncoding(
    L=16,
    F=2,
    input_dim=3,
    include_input=True,
    trainable=True,
    base_resolution=16,
    per_level_scale=1.5,
    log2_hashmap_size=19
).to(device)

dir_encoder_ngp = PositionalEncoding(L=4, include_input=True)

instant_ngp.set_encoder(hash_encoder, dir_encoder_ngp)

# Generate test data (normalized to [0, 1] for hash encoding)
pts = torch.rand(num_rays, num_samples, 3, device=device)
dirs = torch.randn(num_rays, num_samples, 3, device=device)
dirs = F.normalize(dirs, dim=-1)

# Flatten
pts_flat = pts.reshape(-1, 3)
dirs_flat = dirs.reshape(-1, 3)

print(f"\nInput shapes:")
print(f"  Points: {pts_flat.shape}")
print(f"  Directions: {dirs_flat.shape}")

# Forward pass
raw_flat = instant_ngp(pts_flat, dirs_flat)
print(f"InstantNGP output shape (flat): {raw_flat.shape}")

# Reshape
raw = raw_flat.reshape(num_rays, num_samples, 4)
print(f"InstantNGP output shape (reshaped): {raw.shape}")

expected_shape = (num_rays, num_samples, 4)
assert raw.shape == expected_shape, f"❌ Expected {expected_shape}, got {raw.shape}"
print(f"✅ Output shape correct: {raw.shape}")

print(f"\nOutput value ranges:")
print(f"  RGB: [{raw[..., :3].min():.4f}, {raw[..., :3].max():.4f}]")
print(f"  Density: [{raw[..., 3].min():.4f}, {raw[..., 3].max():.4f}]")

print(f"\nTotal parameters: {sum(p.numel() for p in instant_ngp.parameters()):,}")


Input shapes:
  Points: torch.Size([65536, 3])
  Directions: torch.Size([65536, 3])
InstantNGP output shape (flat): torch.Size([65536, 4])
InstantNGP output shape (reshaped): torch.Size([1024, 64, 4])
✅ Output shape correct: torch.Size([1024, 64, 4])

Output value ranges:
  RGB: [0.3522, 0.5660]
  Density: [0.0000, 0.0000]

Total parameters: 16,787,667


In [20]:
from utils.volumerendering import VolumeRenderer
print("\n" + "="*80)
print("Test 3: VolumeRenderer Integration")
print("="*80)

renderer = VolumeRenderer(
    act_fn=F.relu,
    white_bkgd=False,
    raw_noise_std=0.0
).to(device)

# Generate z_vals and rays_d
z_vals = torch.linspace(2.0, 6.0, num_samples, device=device)
z_vals = z_vals.expand(num_rays, num_samples)

rays_d = torch.randn(num_rays, 3, device=device)
rays_d = F.normalize(rays_d, dim=-1)

print(f"\nRenderer inputs:")
print(f"  raw: {raw.shape}")
print(f"  z_vals: {z_vals.shape}")
print(f"  rays_d: {rays_d.shape}")

# Render
outputs = renderer(raw, z_vals, rays_d)

print(f"\nRenderer outputs:")
for key, val in outputs.items():
    print(f"  {key}: {val.shape}")

print(f"\nRendered values:")
print(f"  RGB range: [{outputs['rgb'].min():.4f}, {outputs['rgb'].max():.4f}]")

# Filter out infinite depths for display
valid_depths = outputs['depth'][outputs['depth'] < 1e5]
if valid_depths.numel() > 0:
    print(f"  Depth range: [{valid_depths.min():.4f}, {valid_depths.max():.4f}]")
else:
    print(f"  Depth: All rays empty (no hits)")

print(f"  Acc range: [{outputs['acc'].min():.4f}, {outputs['acc'].max():.4f}]")
print(f"  Weights sum per ray (mean): {outputs['weights'].sum(dim=-1).mean():.4f}")

print("\n✅ VolumeRenderer integration successful!")


Test 3: VolumeRenderer Integration

Renderer inputs:
  raw: torch.Size([1024, 64, 4])
  z_vals: torch.Size([1024, 64])
  rays_d: torch.Size([1024, 3])

Renderer outputs:
  rgb: torch.Size([1024, 3])
  disp: torch.Size([1024, 1])
  acc: torch.Size([1024, 1])
  weights: torch.Size([1024, 64])
  depth: torch.Size([1024, 1])

Rendered values:
  RGB range: [0.0000, 0.0000]
  Depth: All rays empty (no hits)
  Acc range: [0.0000, 0.0000]
  Weights sum per ray (mean): 0.0000

✅ VolumeRenderer integration successful!


In [21]:
print("\n" + "="*80)
print("Test 4: Full Rendering Pipeline")
print("="*80)

from utils.sampler import StratifiedSampler

# Create sampler
sampler = StratifiedSampler(N_samples=64, perturb=1.0, lindisp=False)

# Generate test rays
test_rays = 512
rays_o = torch.randn(test_rays, 3, device=device)
rays_d = torch.randn(test_rays, 3, device=device)
rays_d = F.normalize(rays_d, dim=-1)

bounds = torch.tensor([[2.0, 6.0]], device=device).expand(test_rays, 2)

print(f"\nPipeline inputs:")
print(f"  rays_o: {rays_o.shape}")
print(f"  rays_d: {rays_d.shape}")
print(f"  bounds: {bounds.shape}")

# Step 1: Sample points along rays
pts, z_vals = sampler(rays_o, rays_d, bounds, zvals_only=False)
print(f"\n✅ Step 1 - Stratified Sampling:")
print(f"  pts: {pts.shape}")
print(f"  z_vals: {z_vals.shape}")

# Step 2: Prepare network inputs
pts_normalized = (pts - pts.min()) / (pts.max() - pts.min() + 1e-8)
pts_flat = pts_normalized.reshape(-1, 3)

dirs_expanded = rays_d[:, None, :].expand_as(pts)
dirs_flat = dirs_expanded.reshape(-1, 3)

print(f"\n✅ Step 2 - Prepare inputs:")
print(f"  pts_flat: {pts_flat.shape}")
print(f"  dirs_flat: {dirs_flat.shape}")

# Step 3: Network forward pass
with torch.no_grad():
    raw_flat = instant_ngp(pts_flat, dirs_flat)
    raw = raw_flat.reshape(test_rays, 64, 4)

print(f"\n✅ Step 3 - Network inference:")
print(f"  raw: {raw.shape}")
print(f"  RGB range: [{raw[..., :3].min():.4f}, {raw[..., :3].max():.4f}]")
print(f"  Density range: [{raw[..., 3].min():.4f}, {raw[..., 3].max():.4f}]")

# Step 4: Volume rendering
outputs = renderer(raw, z_vals, rays_d)

print(f"\n✅ Step 4 - Volume rendering:")
print(f"  rgb: {outputs['rgb'].shape}")
print(f"  depth: {outputs['depth'].shape}")
print(f"  acc: {outputs['acc'].shape}")

# Calculate statistics
mean_acc = outputs['acc'].mean().item()
valid_rays = (outputs['acc'] > 0.01).sum().item()

print(f"\nRendering statistics:")
print(f"  Mean accumulation: {mean_acc:.4f}")
print(f"  Valid rays (acc > 0.01): {valid_rays}/{test_rays} ({100*valid_rays/test_rays:.1f}%)")
print(f"  RGB mean: [{outputs['rgb'].mean(dim=0)}")

print("\n✅ Full pipeline successful!")


Test 4: Full Rendering Pipeline

Pipeline inputs:
  rays_o: torch.Size([512, 3])
  rays_d: torch.Size([512, 3])
  bounds: torch.Size([512, 2])

✅ Step 1 - Stratified Sampling:
  pts: torch.Size([512, 64, 3])
  z_vals: torch.Size([512, 64])

✅ Step 2 - Prepare inputs:
  pts_flat: torch.Size([32768, 3])
  dirs_flat: torch.Size([32768, 3])

✅ Step 3 - Network inference:
  raw: torch.Size([512, 64, 4])
  RGB range: [0.3721, 0.5444]
  Density range: [0.0000, 0.0000]

✅ Step 4 - Volume rendering:
  rgb: torch.Size([512, 3])
  depth: torch.Size([512, 1])
  acc: torch.Size([512, 1])

Rendering statistics:
  Mean accumulation: 0.0000
  Valid rays (acc > 0.01): 0/512 (0.0%)
  RGB mean: [tensor([0., 0., 0.])

✅ Full pipeline successful!


In [24]:
print("\n" + "="*80)
print("Test 5: Architecture Comparison")
print("="*80)

import time

architectures = {
    'Standard MLP': mlp,
    'MipMLP': mip_mlp,
    'InstantNGP': instant_ngp
}

test_pts = torch.rand(1024, 3, device=device)
test_dirs = torch.randn(1024, 3, device=device)
test_dirs = F.normalize(test_dirs, dim=-1)
test_cov = torch.randn(1024, 3, device=device).abs() * 0.01

# Prepare inputs for each architecture
pos_enc = PositionalEncoding(L=10, include_input=True)
dir_enc = PositionalEncoding(L=4, include_input=True)
test_pts_encoded = pos_enc(test_pts)
test_dirs_encoded = dir_enc(test_dirs)
mlp_input = torch.cat([test_pts_encoded, test_dirs_encoded], dim=-1)

print("\n" + "-"*80)
print("Architecture Stats:")
print("-"*80)

for name, model in architectures.items():
    params = sum(p.numel() for p in model.parameters())
    
    # Warmup
    with torch.no_grad():
        if name == 'Standard MLP':
            _ = model(mlp_input[:100])
        elif name == 'MipMLP':
            _ = model(test_pts[:100], test_cov[:100], test_dirs[:100])
        else:
            _ = model(test_pts[:100], test_dirs[:100])
    
    # Timing
    start = time.time()
    with torch.no_grad():
        for _ in range(10):
            if name == 'Standard MLP':
                output = model(mlp_input)
            elif name == 'MipMLP':
                output = model(test_pts, test_cov, test_dirs)
            else:
                output = model(test_pts, test_dirs)
    elapsed = (time.time() - start) / 10 * 1000  # ms
    
    print(f"\n{name}:")
    print(f"  Parameters: {params:,}")
    print(f"  Forward pass: {elapsed:.2f} ms")
    print(f"  Output shape: {output.shape}")



Test 5: Architecture Comparison

--------------------------------------------------------------------------------
Architecture Stats:
--------------------------------------------------------------------------------

Standard MLP:
  Parameters: 595,844
  Forward pass: 1.71 ms
  Output shape: torch.Size([1024, 4])

MipMLP:
  Parameters: 595,844
  Forward pass: 2.06 ms
  Output shape: torch.Size([1024, 4])

InstantNGP:
  Parameters: 16,787,667
  Forward pass: 5.05 ms
  Output shape: torch.Size([1024, 4])


In [28]:
print("\n" + "="*80)
print("Testing Improved InstantNGP")
print("="*80)

shapes_to_test = [
    (1024, 3),           # Flat: (N, 3) → (N, 4)
    (32, 32, 3),         # Image-like: (H, W, 3) → (H, W, 4)
    (512, 64, 3),        # Ray-samples: (N_rays, N_samples, 3) → (N_rays, N_samples, 4)
]

ngp = InstantNGP(
    use_embed=True,
    netchunk=1024*32
).to(device)

print(f"\nTotal parameters: {sum(p.numel() for p in ngp.parameters()):,}")

for shape in shapes_to_test:
    pts = torch.rand(*shape, device=device)
    dirs = torch.randn(*shape, device=device)
    dirs = F.normalize(dirs, dim=-1)
    
    print(f"\nTesting shape: {shape}")
    print(f"  Input: {pts.shape}")
    
    with torch.no_grad():
        output = ngp(pts, dirs)
    
    # Expected shape: replace last dim (3) with (4)
    expected_shape = shape[:-1] + (4,)  # ← FIX ICI
    print(f"  Output: {output.shape}")
    print(f"  Expected: {expected_shape}")
    
    assert output.shape == expected_shape, \
        f"Shape mismatch! Got {output.shape}, expected {expected_shape}"
    print(f"✅ Shape test passed")

print("\n" + "="*80)
print("✅ All shape tests passed!")
print("="*80)

large_pts = torch.rand(100000, 3, device=device)  # 100K points
large_dirs = torch.randn(100000, 3, device=device)
large_dirs = F.normalize(large_dirs, dim=-1)

with torch.no_grad():
    output = ngp(large_pts, large_dirs)

print(f"✅ Processed {large_pts.shape[0]:,} points")
print(f"   Output shape: {output.shape}")
print(f"   RGB range: [{output[..., :3].min():.4f}, {output[..., :3].max():.4f}]")
print(f"   Density range: [{output[..., 3].min():.4f}, {output[..., 3].max():.4f}]")

# Test 3: Custom encoders
print("\n" + "-"*80)
print("Testing custom encoder override")
print("-"*80)

from utils.encoding import HashEncoding, PositionalEncoding

custom_hash = HashEncoding(
    L=12, F=4, input_dim=3, include_input=True
).to(device)

custom_dir = PositionalEncoding(L=6, include_input=True)

ngp_custom = InstantNGP(use_embed=True).to(device)
ngp_custom.set_encoder(custom_hash, custom_dir)

test_pts = torch.rand(512, 64, 3, device=device)
test_dirs = torch.randn(512, 64, 3, device=device)
test_dirs = F.normalize(test_dirs, dim=-1)


print("\n" + "="*80)
print("✅ All InstantNGP tests passed!")
print("="*80)


Testing Improved InstantNGP

Total parameters: 16,787,667

Testing shape: (1024, 3)
  Input: torch.Size([1024, 3])
  Output: torch.Size([1024, 4])
  Expected: (1024, 4)
✅ Shape test passed

Testing shape: (32, 32, 3)
  Input: torch.Size([32, 32, 3])
  Output: torch.Size([32, 32, 4])
  Expected: (32, 32, 4)
✅ Shape test passed

Testing shape: (512, 64, 3)
  Input: torch.Size([512, 64, 3])
  Output: torch.Size([512, 64, 4])
  Expected: (512, 64, 4)
✅ Shape test passed

✅ All shape tests passed!
✅ Processed 100,000 points
   Output shape: torch.Size([100000, 4])
   RGB range: [0.3770, 0.6132]
   Density range: [0.0000, 0.0187]

--------------------------------------------------------------------------------
Testing custom encoder override
--------------------------------------------------------------------------------

✅ All InstantNGP tests passed!
